# genesiss-9b — finetune on Text-to-CadQuery (Kaggle)

**Qwen 3.5 family** — Unsloth recommends **bf16 / 16-bit LoRA** (not 4-bit QLoRA) for Qwen 3.5. The model loads with `load_in_16bit=True, load_in_4bit=False`.

Runs on Kaggle. Recommended accelerator: T4 ×2 (tight — Colab A100/H100 is faster).

Pattern
1. Install deps + log in to HF.
2. Pull the latest checkpoint from `HUB_CKPT_REPO` if one exists.
3. Train with SFTTrainer — every save fires a callback that async-uploads the new checkpoint.
4. On finish, push merged 16-bit + GGUF Q4_K_M to `HUB_FINAL_REPO`.


## 1 · Install

In [ ]:
# Install Unsloth — Kaggle.
# Note: Kaggle T4×2 sessions expose 2 GPUs, but a vanilla notebook only
# uses one. To actually use both, save this code as train.py and run
# `!torchrun --nproc_per_node=2 train.py` — Unsloth auto-enables DDP
# when launched under torchrun. The notebook-as-is uses GPU 0 only.
import locale
locale.getpreferredencoding = lambda: "UTF-8"
!pip install --quiet --upgrade pip
!pip install --quiet --upgrade unsloth
!pip install --quiet --upgrade --no-deps --force-reinstall \
    "unsloth @ git+https://github.com/unslothai/unsloth.git" \
    "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
# See Colab install cell for what these add (FA2 on Ampere/Hopper, FA3 on Hopper).
!pip install --quiet --upgrade "huggingface_hub>=0.25" tomli_w kernels
!pip install --quiet --no-build-isolation flash-attn || echo "[install] flash-attn skipped (no wheel for this runtime)"


## 2 · Authenticate with Hugging Face

In [ ]:
import os

def _load_hf_token() -> str | None:
    # 1) Already in the environment (VS Code `.env`, shell export,
    #    runtime env var passed in by an orchestrator, etc).
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    # 2) Colab secret (browser UI: 🔑 sidebar → Add secret).
    try:
        from google.colab import userdata  # type: ignore[import-not-found]
        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok
    except Exception:
        pass
    # 3) Kaggle secret (Add-ons → Secrets).
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore[import-not-found]
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    return None

token = _load_hf_token()
if not token:
    raise RuntimeError(
        "HF_TOKEN not found. Set it via ONE of:\n"
        "  - Colab browser: 🔑 sidebar → Add secret `HF_TOKEN` (works for VS Code/Jupyter too once set)\n"
        "  - Kaggle: Add-ons → Secrets → add `HF_TOKEN`\n"
        "  - Shell:  export HF_TOKEN=hf_xxx  (then restart the kernel)"
    )
os.environ["HF_TOKEN"] = token

from huggingface_hub import login
login(token=token, add_to_git_credential=False)
print("[hf] logged in")


## 3 · Vendor the shared helpers

In [ ]:
# Vendor training/shared/* into the runtime so we can `from shared import ...`.
# If you've cloned the repo into /kaggle/working, you can skip this cell and just
# `import sys; sys.path.insert(0, "/kaggle/working/genesiss/training")`.
import os, urllib.request, sys
SHARED_BASE = "https://raw.githubusercontent.com/numinousmuses/genesiss/main/training/shared"
TARGET = "/kaggle/working/shared"
os.makedirs(TARGET, exist_ok=True)
for fname in ["__init__.py", "data_loader.py", "format.py", "checkpoint.py"]:
    url = f"{SHARED_BASE}/{fname}"
    try:
        urllib.request.urlretrieve(url, f"{TARGET}/{fname}")
    except Exception as e:
        print(f"[warn] could not fetch {fname} from GitHub ({e}). "
              "Paste the file contents manually if running before the repo is public.")
sys.path.insert(0, "/kaggle/working")


## 4 · Run config

In [ ]:
# ---- run config ---------------------------------------------------------
VARIANT       = "genesiss-9b"
BASE_MODEL    = "unsloth/Qwen3.5-9B"
FAMILY        = "qwen3.5"                     # qwen3.5 or gpt-oss

# HF user (override if forking).
HF_USER       = "numinousmuses"

# Where checkpoints live on the Hub.  Each `checkpoint-XXXX` folder is a
# full Trainer save (model.safetensors, optimizer.pt, scheduler.pt,
# trainer_state.json, etc.) so we can resume mid-step.
HUB_CKPT_REPO  = f"{HF_USER}/{VARIANT}-checkpoints"
# Final merged model / GGUF goes here.
HUB_FINAL_REPO = f"{HF_USER}/{VARIANT}"

MAX_SEQ_LENGTH               = 4096
PER_DEVICE_TRAIN_BATCH_SIZE  = 1
GRADIENT_ACCUMULATION_STEPS  = 16
LORA_R                       = 16
LORA_ALPHA                   = 16
LEARNING_RATE                = 0.0002
SAVE_STEPS                   = 100
# See Variant dataclass for the meaning of values here.
# True = HF standard (recommended for 4B/9B/20B on 80GB).
# "unsloth" = aggressive offload mode (for 27B-class memory pressure).
# False = no checkpointing (fastest, ~2× memory of True).
GRADIENT_CHECKPOINTING       = True
# Unsloth LoRA guide says 1-3 epochs with diminishing returns past 3.
# One epoch of the combined ~332k-row dataset is already a LOT of
# supervision — typical SFT runs use far less. Bump to 2 only if you
# have headroom in your compute-units budget after a 1-epoch baseline.
NUM_EPOCHS                   = 1

# Sanity-check knob. Set to e.g. 5000 for a quick 10-minute run to
# verify the pipeline before spending compute units on the full
# ~380k-row combined train split. Set back to None for production.
MAX_TRAIN                    = None
MAX_EVAL                     = 2000

# Hard wall-clock budget per session. Colab's idle kick is ~24h on
# Pro+; 23h leaves headroom for the final save + HF upload. When the
# budget expires the trainer saves cleanly and stops — re-running the
# notebook resumes from the last checkpoint with a fresh budget.
MAX_TRAIN_SECONDS            = 23 * 3600

# Attention implementation — auto-detected by what's actually
# importable AND what the current GPU supports. Two things that
# tripped us up on Colab:
#   (a) flash-attn isn't pre-installed even on FA2-capable GPUs
#   (b) gpt-oss only accepts FA3/FA4, and the FA3 kernel
#       (kernels-community/vllm-flash-attn3) requires Hopper —
#       fails at runtime on Ampere with "S aux is currently only
#       supported for Hopper GPUs".
# So we probe both: package availability AND compute capability.
def _auto_attn_for_family(family: str) -> str | None:
    try:
        import torch
        if torch.cuda.is_available():
            major = torch.cuda.get_device_capability(0)[0]
        else:
            major = 0
    except Exception:
        major = 0
    is_hopper_or_newer = major >= 9  # H100=9.0, B100/B200=10.0

    try:
        import flash_attn  # noqa: F401
        fa2_ok = True
    except Exception:
        fa2_ok = False
    try:
        import kernels  # noqa: F401
        kernels_ok = True
    except Exception:
        kernels_ok = False

    if family == "gpt-oss":
        # FA3 needs both the kernel package AND a Hopper-class GPU.
        # Anything else → eager (slow but always loads).
        if kernels_ok and is_hopper_or_newer:
            return "kernels-community/vllm-flash-attn3"
        return "eager"
    # Qwen 3.5: FA2 if the package is importable, else xformers (None).
    return "flash_attention_2" if fa2_ok else None

ATTN_IMPLEMENTATION          = _auto_attn_for_family(FAMILY)
print(f"[attn] picked {ATTN_IMPLEMENTATION!r} for {FAMILY}")

# Local dir Trainer writes checkpoints to (then async-pushed to HUB_CKPT_REPO).
OUTPUT_DIR = f"./outputs/{VARIANT}"


## 5 · Resume from Hub (if any)

In [ ]:
# Resume from the Hub if a prior session uploaded a checkpoint.
# `pull_latest_checkpoint` returns the local path, or None.
from shared.checkpoint import pull_latest_checkpoint
resumed_path = pull_latest_checkpoint(HUB_CKPT_REPO, OUTPUT_DIR, token=os.environ["HF_TOKEN"])
print("resumed from:", resumed_path)


## 6 · Load base model + attach LoRA

In [ ]:
from unsloth import FastLanguageModel

# Pass attn_implementation only when ATTN_IMPLEMENTATION is set (None
# means: let transformers pick the best available — FA2 → SDPA → eager).
_load_kwargs = dict(
    model_name = BASE_MODEL,
            max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
if ATTN_IMPLEMENTATION is not None:
    _load_kwargs["attn_implementation"] = ATTN_IMPLEMENTATION

model, tokenizer = FastLanguageModel.from_pretrained(**_load_kwargs)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = LORA_ALPHA,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = GRADIENT_CHECKPOINTING,
    random_state = 3407,
    max_seq_length = MAX_SEQ_LENGTH,
)


## 7 · Load dataset + apply chat template

In [ ]:
# Load both training sources by default:
#   - ricemonster/NeurIPS11092 (~170k, Text2CAD-derived, license unspecified)
#   - gudo7208/CAD-Coder       (~250k, Apache-2.0, ChatML messages format)
# Both are normalized to the same `messages` column, concatenated, and
# shuffled with a fixed seed (see training/shared/data_loader.py).
# MAX_TRAIN/MAX_EVAL come from the run-config cell — MAX_TRAIN=None means
# use the full combined train set (~380k rows after concatenation).
from shared.data_loader import load_dataset_dict
from shared.format import text_field

ds = load_dataset_dict(max_train=MAX_TRAIN, max_eval=MAX_EVAL)
ds = ds.map(lambda b: {
    "text": [
        tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
        for m in b["messages"]
    ]
}, batched=True, remove_columns=["messages"])

print(ds)
print("--- example ---")
print(ds["train"][0]["text"][:1200])


## 8 · Build trainer (with async Hub-upload callback)

In [ ]:
from trl import SFTTrainer, SFTConfig
from shared.checkpoint import make_hub_callback, make_time_budget_callback

hub_cb     = make_hub_callback(HUB_CKPT_REPO, OUTPUT_DIR, token=os.environ["HF_TOKEN"])
budget_cb  = make_time_budget_callback(max_seconds=MAX_TRAIN_SECONDS)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds["train"],
    eval_dataset = ds["validation"],
    args = SFTConfig(
        output_dir = OUTPUT_DIR,
        max_seq_length = MAX_SEQ_LENGTH,
        dataset_text_field = "text",
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        # Unsloth LoRA hyperparam guide: warmup 5-10% of total steps,
        # weight_decay 0.01, linear scheduler, adamw_8bit.
        warmup_ratio = 0.05,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        # Save every SAVE_STEPS — the HubCheckpointCallback async-pushes
        # each checkpoint to HUB_CKPT_REPO so we can resume next session.
        save_strategy = "steps",
        save_steps = SAVE_STEPS,
        save_total_limit = 2,
        # Newer transformers renamed evaluation_strategy → eval_strategy.
        eval_strategy = "steps",
        eval_steps = SAVE_STEPS,
        report_to = "none",
        dataset_num_proc = 2,
    ),
    callbacks = [hub_cb, budget_cb],
)


## 9 · Train

In [ ]:
# `resume_from_checkpoint=True` makes the Trainer look in OUTPUT_DIR for the
# latest checkpoint-XXXX (the one we just pulled, if any).
trainer_stats = trainer.train(resume_from_checkpoint=True if resumed_path else False)
print(trainer_stats)


## 10 · Smoke test

In [ ]:
# Quick smoke test on the trained adapter — does it emit CadQuery?
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content":
        "You are Genesiss, a CAD assistant. Output ONLY CADQuery python."},
    {"role": "user", "content": "a 30mm cube with a 10mm hole through the centre"},
]
ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt"
).to(model.device)
out = model.generate(ids, max_new_tokens=400, do_sample=False)
print(tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True))


## 11 · Export (merged 16-bit + GGUF for Ollama)

In [ ]:
# 1) Push merged 16-bit weights for HF inference / vLLM / transformers.
model.push_to_hub_merged(
    HUB_FINAL_REPO, tokenizer,
    save_method = "merged_16bit",
    token = os.environ["HF_TOKEN"],
)

# 2) GGUF + auto-generated Modelfile (Unsloth writes both into the
#    same directory; the Modelfile bakes in the trained chat template).
# If save_pretrained_gguf crashes mid-export, drop
# maximum_memory_usage to 0.5 per Unsloth's troubleshooting guide:
#     model.save_pretrained_gguf(GGUF_DIR, tokenizer,
#         quantization_method="q4_k_m", maximum_memory_usage=0.5)
GGUF_DIR = f"./gguf-{VARIANT}"
model.save_pretrained_gguf(
    GGUF_DIR, tokenizer,
    quantization_method = "q4_k_m",
)
# Optional: peek at the auto-generated Modelfile.
print(open(f"{GGUF_DIR}/Modelfile").read())

# 3) Push GGUF + Modelfile to the Hub so `genesiss models pull` can
#    download them locally and `ollama create` from there.
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
api.upload_folder(
    folder_path = GGUF_DIR,
    repo_id = HUB_FINAL_REPO,
    repo_type = "model",
    path_in_repo = "gguf",
    commit_message = "GGUF Q4_K_M + Modelfile",
)

print(f"\nDone. To use locally:")
print(f"  genesiss models pull {VARIANT}")
print(f"  ollama run {VARIANT}")
